# HireMind: retrain M0 with 512-token inputs (Senior Project II)

**Why:** the classifier reads only the first 256 tokens of each CV and job description together, while CVs in the dataset have a median of about 1,563 tokens. The midterm report (Ch. 5) commits to *512-token retraining* to reduce this truncation.

**What changes:** only the input length (256 → 512). The model (BERT-Tiny M0), learning rate (1e-4), seeds (42, 43, 44), split and every other setting stay exactly as in the DL paper.

**Rules:** choose between 256 and 512 on **validation only**. Open the test split once, at the end, only if 512 is chosen.

Runtime: Runtime → Change runtime type → **T4 GPU**. Then run the cells in order. Total time is about 1 hour.

In [ ]:
# 1. Check the GPU and get the code
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/Hams770/HireMind-DL.git
%cd HireMind-DL
!git log --oneline -1

In [ ]:
# 2. Install the pinned versions (keeps Colab's own GPU build of torch)
!grep -v '^torch' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

In [ ]:
# 3. Switch to 512 tokens (all outputs go to runs_L512/, data/cache_L512/ and outputs/L512/)
%env HIREMIND_MAX_LEN=512
!python scripts/smoke_test.py

In [ ]:
# 4. Download the pinned dataset and tokenize at 512 (validation and train only; test is not touched yet)
!python scripts/prepare_data.py --no-test

In [ ]:
# 5. Train M0 at 512 tokens, three seeds (about 10 minutes each on a T4)
!python scripts/train.py --model M0

In [ ]:
# 6. Compare 512 with the published 256 results on VALIDATION
!python scripts/evaluate.py --split validation --models M0
import pandas as pd
new = pd.read_csv('outputs/L512/validation_per_seed.csv')
old = pd.read_csv('assignment_3/item3_evidence/reports/validation/per_seed.csv')
old = old[old.model == 'M0'][['seed', 'macro_f1', 'accuracy']]
cmp = old.merge(new[['seed', 'macro_f1', 'accuracy']], on='seed', suffixes=('_256', '_512'))
print(cmp.round(2).to_string(index=False))
print('\nMean validation macro-F1: 256 =', round(cmp.macro_f1_256.mean(), 2), '| 512 =', round(cmp.macro_f1_512.mean(), 2))
CHOOSE_512 = cmp.macro_f1_512.mean() > cmp.macro_f1_256.mean()
print('Decision on validation:', '512 is better -> run the next cell' if CHOOSE_512 else '256 stays -> skip the test cell, report the validation result')

**Only if the previous cell chose 512.** This opens the held-out test once and measures CPU latency.

In [ ]:
# 7. Held-out test for the chosen 512 model + CPU latency (1 thread, batch 1)
assert CHOOSE_512, 'Validation did not choose 512: do not open the test split'
!python scripts/prepare_data.py
!python scripts/evaluate.py --split test --models M0 --benchmark

In [ ]:
# 8. Save everything to Google Drive and send the zip to Hams
from google.colab import drive
drive.mount('/content/drive')
!zip -qr /content/drive/MyDrive/HireMind_M0_L512.zip runs_L512 outputs/L512
!ls -la /content/drive/MyDrive/HireMind_M0_L512.zip
!sha256sum runs_L512/*/weights.pt